# Tagalog/Taglish Lexicon and Labeling Template

Builds the custom VADER lexicon (English + Tagalog/Taglish words, negations, intensifiers) and exports the training records as `extracted_sentiment_data.csv`. That export is the template the group hand-labeled into **`datasets/final_manual_labels.csv`**.

**Input:** `final_training_data.csv` (and `final_cleaned_combined_data.csv` for the 50 extra records in the last cells).
You don't need to re-run this to get the labels; they are already in `final_manual_labels.csv`.

In [ ]:
import pandas as pd
import re
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Download VADER lexicon
nltk.download('vader_lexicon', quiet=True)

# Load the training dataset
import glob, os

# Looks in datasets/ (running locally from the repo) and /content/ (Colab uploads).
# Also finds Colab-renamed copies like "final_training_data (1).csv".
DATA_DIRS = ["datasets", "/content/datasets", "/content", "."]

def find_file(name, required=True):
    stem, ext = os.path.splitext(name)
    for d in DATA_DIRS:
        exact = os.path.join(d, name)
        if os.path.exists(exact):
            return exact
        hits = sorted(glob.glob(os.path.join(d, stem + " (*)" + ext)))
        if hits:
            return hits[-1]
    if required:
        raise FileNotFoundError(f"Upload {name} to /content/ (Colab) or keep it in datasets/ (local).")
    return None

file_path = find_file('final_training_data.csv')
try:
    df = pd.read_csv(file_path)
    print("Dataset loaded successfully!")
    display(df.head())
except Exception as e:
    print(f"Error loading dataset: {e}")

In [ ]:
import re
import pandas as pd
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

nltk.download('vader_lexicon', quiet=True)

# Initialize VADER Sentiment Analyzer
sia = SentimentIntensityAnalyzer()

# FIX: remove Tagalog words that VADER scores as English slang.
# "po" is -2.6 in VADER, which made polite comments like "thanks po" negative.
for word in ['po', 'ho', 'ha', 'nag', 'wag']:
    sia.lexicon.pop(word, None)

# Define Tagalog/Taglish sentiment additions
tagalog_lexicon = {
    # Positive words
    'maganda': 2.0,
    'mabuti': 1.5,
    'masaya': 2.0,
    'gusto': 1.5,
    # FIX: 'mahal' removed. In this topic it usually means "expensive" (mahal ang tuition).
    'salamat': 1.5,
    'lodi': 2.0,
    'petmalu': 2.0,
    'werpa': 2.0,
    'astig': 1.8,
    'bilib': 1.5,
    'paborito': 1.8,
    'sulit': 2.0,
    # Added: words common in education discussions
    'magaling': 2.0,
    'mahusay': 2.0,
    'mabait': 1.5,
    'matalino': 1.8,
    'masipag': 1.5,
    'maayos': 1.5,
    'saludo': 2.0,
    'tagumpay': 2.0,

    # Negative words
    'pangit': -2.0,
    'masama': -2.0,
    'galit': -2.0,
    'lungkot': -1.5,
    'ayaw': -1.5,
    'inis': -1.5,
    'bobo': -2.5,
    'tanga': -2.5,
    'gago': -3.0,
    'kupal': -2.5,
    'sayang': -1.0,
    'mahirap': -1.0,
    'bulok': -2.0,
    # Added: words common in education discussions
    'hirap': -1.5,
    'kulang': -1.5,
    'kakulangan': -1.5,
    'kawawa': -1.5,
    'bagsak': -2.0,
    'palpak': -2.0,
    'kurakot': -2.5,
    'korap': -2.5,
    'problema': -1.5,
    'tamad': -1.5,
    'bastos': -2.0,
    'mali': -1.5,
    'walangkwenta': -2.5,   # from the phrase "walang kwenta"
    'walangsilbi': -2.5     # from the phrase "walang silbi"
}

# Update VADER lexicon with Tagalog words
sia.lexicon.update(tagalog_lexicon)

# FIX: roots and prefixes so inflected words are matched
# (napakaganda, nakakalungkot, nakakainis). Roots only count after a prefix is removed.
tagalog_roots = {
    'ganda': 2.0, 'galing': 2.0, 'husay': 2.0, 'bait': 1.5, 'saya': 2.0, 'tuwa': 1.8,
    'sama': -2.0, 'hiya': -2.0, 'awa': -1.5, 'dismaya': -2.0
}
tagalog_prefixes = {      # prefix: multiplier
    'napaka': 1.8,        # napaka- is an intensifier attached to the word
    'pinaka': 1.5,
    'nakaka': 1.0,
    'naka': 1.0,
    'ma': 1.0,
    'na': 1.0
}
tagalog_phrases = {
    'walang kwenta': 'walangkwenta',
    'walang kuwenta': 'walangkwenta',
    'walang silbi': 'walangsilbi'
}

# Define custom lists for negations and intensifiers
# FIX: English negations added. Looking words up directly skips VADER's own
# negation rules, so "not good" was being scored as positive.
tagalog_negations = {'hindi', 'hinde', 'di', 'wala', 'huwag', 'wag'}
english_negations = {'not', 'no', 'never', 'without', 'cannot', 'cant', 'dont', 'doesnt',
                     'didnt', 'isnt', 'arent', 'wasnt', 'werent', 'wont', 'wouldnt',
                     'shouldnt', 'couldnt', 'aint'}
all_negations = tagalog_negations | english_negations

tagalog_intensifiers = {
    'sobra': 1.5,     # multiplies next word score
    'napaka': 1.8,
    'grabe': 1.8,
    'masyado': 1.5,   # FIX: was 'masyadong'; the -ng ending is now handled automatically
    'talaga': 1.3,
    'super': 1.5,
    'very': 1.5,
    'really': 1.3
}

# FIX: negations and intensifiers now expire instead of waiting for the next
# sentiment word no matter how far away it is.
NEGATION_WINDOW = 3       # affects sentiment words up to 3 words after
INTENSIFIER_WINDOW = 2    # affects sentiment words up to 2 words after

def strip_ligature(token):
    # magandang -> maganda, sobrang -> sobra, walang -> wala
    forms = [token]
    if len(token) > 4 and token.endswith('ng'):
        forms.append(token[:-2])
    if len(token) > 4 and token.endswith('g'):
        forms.append(token[:-1])
    return forms

def base_form(token, word_set):
    for form in strip_ligature(token):
        if form in word_set:
            return form
    return None

def get_word_score(token):
    # Returns (score, multiplier from prefix)
    if token in sia.lexicon:
        return sia.lexicon[token], 1.0
    for form in strip_ligature(token):
        if form in tagalog_lexicon:
            return tagalog_lexicon[form], 1.0
        for prefix, prefix_multiplier in tagalog_prefixes.items():
            root = form[len(prefix):]
            if form.startswith(prefix) and len(root) >= 3:
                if root in tagalog_roots:
                    return tagalog_roots[root], prefix_multiplier
                if root in tagalog_lexicon and len(root) >= 4:
                    return tagalog_lexicon[root], prefix_multiplier
    return 0.0, 1.0

def custom_sentiment_score(text):
    if not isinstance(text, str):
        return 0.0

    text = text.lower().replace("'", "").replace("\u2019", "")
    for phrase, joined in tagalog_phrases.items():
        text = text.replace(phrase, joined)
    tokens = re.findall(r"[a-z\u00f1]+", text)

    word_scores = []
    negate_left = 0
    multiplier = 1.0
    multiplier_left = 0

    for token in tokens:
        # Check if current token is a negation
        if base_form(token, all_negations):
            negate_left = NEGATION_WINDOW
            continue

        # Check if current token is an intensifier
        intensifier = base_form(token, tagalog_intensifiers)
        if intensifier:
            multiplier = tagalog_intensifiers[intensifier]
            multiplier_left = INTENSIFIER_WINDOW
            continue

        # Retrieve base score from the lexicon
        word_score, prefix_multiplier = get_word_score(token)

        # Apply logic
        if word_score != 0.0:
            current_word_score = word_score * prefix_multiplier
            if multiplier_left > 0:
                current_word_score *= multiplier
                multiplier_left = 0   # Reset multiplier
            if negate_left > 0:
                current_word_score = -current_word_score
                negate_left = 0       # Reset negation
            word_scores.append(current_word_score)
        else:
            negate_left = max(0, negate_left - 1)
            multiplier_left = max(0, multiplier_left - 1)

    # FIX: average instead of sum, so long articles do not get huge scores
    if not word_scores:
        return 0.0
    return round(sum(word_scores) / len(word_scores), 2)

def text_or_title(row):
    # FIX: records with no text were scored as 0 (Neutral). Fall back to the title.
    text_val = row.get('text')
    if pd.isna(text_val) or str(text_val).strip() == '':
        title_val = row.get('title')
        return title_val if pd.notna(title_val) else ''
    return text_val

print("Lexicon and custom Tagalog/Taglish sentiment logic configured!")

In [ ]:
# Example Demonstration
test_sentences = [
    "Sobra nating gusto ang magandang serbisyo na ito",
    "Hindi maganda ang kulay at bulok ang system",
    "Grabe masaya ako ngayon pero medyo mahirap mag-adjust"
]

for sent in test_sentences:
    print(f"Text: {sent}")
    print(f"Calculated Sentiment Score: {custom_sentiment_score(sent):.2f}\n")

# Checks for the fixes
fix_checks = [
    "thanks po sa drills",
    "the quality of education is not good",
    "hindi ko alam pero maganda ang school",
    "napakaganda ng school",
    "walang kwenta ang deped",
    "mahal ang tuition"
]

print("--- Fix checks ---")
for sent in fix_checks:
    print(f"Text: {sent}")
    print(f"Calculated Sentiment Score: {custom_sentiment_score(sent):.2f}\n")

In [ ]:
# Apply the custom sentiment analysis function to the 'text' column of the training dataset
# FIX: uses the title when text is missing
df['calculated_sentiment_score'] = df.apply(lambda row: custom_sentiment_score(text_or_title(row)), axis=1)

# Categorize sentiment based on the calculated scores
def categorize_sentiment(score):
    if score > 0.5:
        return 'Positive'
    elif score < -0.5:
        return 'Negative'
    else:
        return 'Neutral'

df['sentiment_category'] = df['calculated_sentiment_score'].apply(categorize_sentiment)

# Display summary and sample results
print("Sentiment Categories Distribution:")
print(df['sentiment_category'].value_counts())

print("\nSample records with calculated sentiment scores:")
display(df[['text', 'calculated_sentiment_score', 'sentiment_category']].head(10))

In [ ]:
# Map columns matching your requested headers precisely
export_df = pd.DataFrame()

# 1. Map ID, Source, and URL
export_df['record_id'] = df['record_id']
export_df['source'] = df['platform'] if 'platform' in df.columns else (df['source_name'] if 'source_name' in df.columns else '')
export_df['url'] = df['url'] if 'url' in df.columns else ''

# 2. Map date_posted using 'published_at'
if 'published_at' in df.columns:
    export_df['date_posted'] = df['published_at']
else:
    export_df['date_posted'] = ''

# 3. Clean and map text (falling back to title if text is missing or blank)
def clean_text_field(row):
    text_val = row.get('text')
    title_val = row.get('title')

    # If text is NaN or purely empty whitespace, fall back to title
    if pd.isna(text_val) or str(text_val).strip() == '':
        return title_val if pd.notna(title_val) else ''
    return text_val

export_df['text'] = df.apply(clean_text_field, axis=1)

# 4. Map language directly from the 'detected_language' column
export_df['language'] = df['detected_language'] if 'detected_language' in df.columns else ''

# 5. Map tokens
export_df['tokens'] = df['tokens'] if 'tokens' in df.columns else ''

# Save the refined dataset to the required output path
output_file_path = 'extracted_sentiment_data.csv'
export_df.to_csv(output_file_path, index=False)

print(f"Corrected CSV file successfully created at: {output_file_path}")
print(f"Total exported records: {len(export_df)}")
print(f"Remaining rows with null or empty texts: {export_df['text'].isna().sum() + (export_df['text'].astype(str).str.strip() == '').sum()}")
print("\nPreview of the refined export:")
display(export_df.head(10))

In [ ]:
# Download the labeling template (Colab only)
try:
    from google.colab import files
    files.download('extracted_sentiment_data.csv')
except ImportError:
    print("Running locally: extracted_sentiment_data.csv is in", os.getcwd())

In [ ]:
import pandas as pd
import re

# Load the datasets
df_train = pd.read_csv(find_file('final_training_data.csv'))
df_combined = pd.read_csv(find_file('final_cleaned_combined_data.csv'))

# Normalize helper for accurate comparison
def clean_normalize(t):
    if pd.isna(t):
        return ""
    t = str(t).lower().strip()
    t = re.sub(r'[^a-z0-9\s]', '', t) # remove punctuation
    t = re.sub(r'\s+', ' ', t) # remove duplicate whitespaces
    return t

# Apply normalization to texts
df_train['clean_text'] = df_train['text'].apply(clean_normalize)
df_combined['clean_text'] = df_combined['text'].apply(clean_normalize)

# Find records in combined not in training
train_set = set(df_train['clean_text'])
unique_mask = ~df_combined['clean_text'].isin(train_set) & (df_combined['clean_text'] != '')
df_uniques = df_combined[unique_mask]

# Let's drop exact duplicates on clean_text within these uniques
df_unique_50 = df_uniques.drop_duplicates(subset=['clean_text'], keep='first')

# If it is still more than 50, let's limit it to exactly 50 unique items to match your exact expected threshold
if len(df_unique_50) > 50:
    df_unique_50 = df_unique_50.head(50)

print(f"Final count of unique records: {len(df_unique_50)}")

# Compute custom sentiment score for these 50 records
df_unique_50 = df_unique_50.copy()
df_unique_50['calculated_sentiment_score'] = df_unique_50.apply(lambda row: custom_sentiment_score(text_or_title(row)), axis=1)
df_unique_50['sentiment_category'] = df_unique_50['calculated_sentiment_score'].apply(
    lambda s: 'Positive' if s > 0.5 else ('Negative' if s < -0.5 else 'Neutral')
)

# Map columns to standard output format
output_df = pd.DataFrame()
output_df['record_id'] = df_unique_50['record_id']
output_df['source'] = df_unique_50['platform'].fillna(df_unique_50.get('source_name', ''))
output_df['url'] = df_unique_50['url'].fillna('')
output_df['date_posted'] = df_unique_50.get('published_at', '')
output_df['text'] = df_unique_50['text']
output_df['language'] = df_unique_50.get('detected_language', '')
output_df['tokens'] = df_unique_50.get('tokens', '')
output_df['calculated_sentiment_score'] = df_unique_50['calculated_sentiment_score']
output_df['sentiment_category'] = df_unique_50['sentiment_category']

# Save locally to match formatting
output_path = 'extracted_sentiment_data_50.csv'
output_df.to_csv(output_path, index=False)
print(f"File saved successfully to {output_path}!")

# Display sample
display(output_df.head())

In [ ]:
try:
    from google.colab import files
    files.download('extracted_sentiment_data_50.csv')
except ImportError:
    print("Running locally: extracted_sentiment_data_50.csv is in", os.getcwd())